# Features, and what each one throws away

MichAl Academy, lesson 2.2.

Run each cell with **Shift+Enter**.

A model never sees your data. It sees a table of numbers that somebody decided
to compute, and that decision is worth more than the choice of algorithm. This
notebook measures four versions of that decision on three standard datasets.


## 1. One column nearly solves it, another barely helps

Same 150 flowers as lesson 2.1. Train on one measurement at a time and score
each. Five-fold cross-validation rather than a single split, so the numbers are
not one lucky shuffle.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def accuracy(X, y, model=None):
    """Mean five-fold accuracy. Scaled logistic regression unless told otherwise."""
    model = model or make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_score(model, X, y, cv=CV).mean()


iris = load_iris()
for i, name in enumerate(iris.feature_names):
    print(f"{name:>20}  {accuracy(iris.data[:, [i]], iris.target):.3f}")

print(f"{'all four together':>20}  {accuracy(iris.data, iris.target):.3f}")


Nothing about the model changed between those lines. Hand it sepal width and it
is not much better than a coin; hand it petal width and it is within a whisker
of the full table.

The feature was the whole difference. That is the lesson in one measurement, and
everything below is a consequence of it.


## 2. Every feature you compute throws something away

The digits dataset: 1,797 handwritten digits as 8x8 grids of ink, values 0 to 16.

Worth knowing before you call these pixels raw. According to scikit-learn's own
dataset description, the originals were 32x32 bitmaps, and the 8x8 grid was made
by dividing them into 4x4 blocks and counting the on-pixels in each. What you
are given is already the third rung of somebody else's ladder.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier

digits = load_digits()
X, y = digits.data, digits.target
images = X.reshape(-1, 8, 8)

three = images[np.argmax(y == 3)]
print("one digit, as the model gets it:")
print(three.astype(int))
print()
print("shape:", X.shape, " classes:", len(np.unique(y)))


In [ ]:
# Four ways to describe the same 1,797 images.
rows = images.sum(axis=2)          # ink in each of the 8 rows
cols = images.sum(axis=1)          # ink in each of the 8 columns

representations = {
    "every pixel":            X,
    "ink per row and column": np.hstack([rows, cols]),
    "ink per row":            rows,
    "total ink":              X.sum(axis=1).reshape(-1, 1),
}

baseline = accuracy(X, y, DummyClassifier(strategy="most_frequent"))
print(f"{'representation':>24} {'features':>9} {'accuracy':>9}")
for name, Xr in representations.items():
    print(f"{name:>24} {Xr.shape[1]:>9} {accuracy(Xr, y):>9.3f}")
print(f"{'always guess the mode':>24} {'-':>9} {baseline:>9.3f}")


Read down that column. Each step is a defensible feature choice and each one
costs accuracy.

"Total ink" is the one to sit with. It is a real property of the image, it is
cheap to compute, and it sounds like a feature. Against ten classes where
guessing scores about 0.10 it is worth almost nothing, and this is why.


In [ ]:
ink = X.sum(axis=1)

per_class = pd.DataFrame({
    "average ink": [ink[y == c].mean() for c in range(10)],
    "spread within this digit": [ink[y == c].std() for c in range(10)],
}, index=[f"digit {c}" for c in range(10)])
print(per_class.round(1).to_string())
print()
print(f"spread between the ten averages: {per_class['average ink'].std():.1f}")
print(f"average spread inside one digit: {per_class['spread within this digit'].mean():.1f}")


There it is. The ten averages sit within a few units of one another, while the
ink used by one digit varies several times more than that from sample to sample.
The differences between the classes are smaller than the variation inside them,
so no threshold on this number can separate them.

A feature is not a fact about your data. It is a decision about what to keep,
and the bill arrives as accuracy.


## 3. So why engineer at all? Because the units are a feature too

The wine dataset: 178 wines from one region of Italy, three growers, thirteen
chemical measurements each. Look at the ranges before doing anything else.


In [ ]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier

wine = load_wine()
Xw, yw = wine.data, wine.target

spread = pd.DataFrame({
    "min": Xw.min(axis=0),
    "max": Xw.max(axis=0),
}, index=wine.feature_names).sort_values("max", ascending=False)

print("widest three columns")
print(spread.head(3).round(2).to_string())
print()
print("narrowest three")
print(spread.tail(3).round(2).to_string())


Proline reaches 1,680 while the narrowest columns never leave single digits. Now
train the same model twice, once on those numbers and once after rescaling every
column to a comparable spread.


In [ ]:
print("k-nearest neighbours")
print(f"  raw     {accuracy(Xw, yw, KNeighborsClassifier()):.3f}")
print(f"  scaled  {accuracy(Xw, yw, make_pipeline(StandardScaler(), KNeighborsClassifier())):.3f}")


Same 178 wines, same thirteen measurements, same algorithm. The gap is the
units.

Here is the mechanism, in arithmetic rather than in words. k-nearest neighbours
decides by distance, so ask what each column contributes to the distance between
two particular wines.


In [ ]:
a, b = Xw[0], Xw[120]
gap_raw = (a - b) ** 2
share_raw = gap_raw / gap_raw.sum()

scaled = StandardScaler().fit_transform(Xw)
gap_scaled = (scaled[0] - scaled[120]) ** 2
share_scaled = gap_scaled / gap_scaled.sum()

contrib = pd.DataFrame({
    "raw share": share_raw,
    "scaled share": share_scaled,
}, index=wine.feature_names).sort_values("raw share", ascending=False)
print(contrib.head(4).round(4).to_string())


Before scaling, proline accounts for **99.5%** of the distance between those two
wines. The other twelve measurements are sitting right there in the table and
the model cannot hear them, so "nearest neighbour" means "closest on proline"
and nothing else. After scaling, the largest single contribution is under half
and alcohol has become the column that separates them.

Scaling did not add information. It stopped one column from drowning out the
rest, which is what the scikit-learn preprocessing guide warns about: a feature
whose variance is orders of magnitude larger than the others "might dominate the
objective function and make the estimator unable to learn from other features
correctly".


## 4. Scaling is a property of the model, not a virtue

Try the same comparison with a decision tree.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=0)
print(f"tree, raw     {accuracy(Xw, yw, tree):.3f}")
print(f"tree, scaled  {accuracy(Xw, yw, make_pipeline(StandardScaler(), tree)):.3f}")


Identical to three decimals, and for a reason worth keeping: a tree asks "is
this column above some threshold", and rescaling a column moves the threshold by
exactly as much as it moves the values. The question it answers is unchanged.

So "always scale your features" is not advice, it is a habit that happens to be
harmless. The real statement is that distance-based and gradient-based models
care about scale and tree-based models do not, which is why lessons 2.5 and 2.6
can skip the step entirely.


## 5. When the raw field is not a number at all

Everything above started from numbers. Real tables are full of fields that are
not: a country, a protocol, a device type. Those have to be turned into numbers
before any of this applies, and there are two ways to do it that are not
interchangeable.

Four rows is enough to see the problem.


In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

proto = np.array([["tcp"], ["udp"], ["icmp"], ["tcp"]])

ordinal = OrdinalEncoder().fit_transform(proto)
onehot = OneHotEncoder(sparse_output=False).fit_transform(proto)

print("ordinal encoding")
for p, v in zip(proto.ravel(), ordinal.ravel()):
    print(f"  {p:>5} -> {v:.0f}")
print()
print("one-hot encoding, columns:", OneHotEncoder().fit(proto).categories_[0].tolist())
for p, v in zip(proto.ravel(), onehot):
    print(f"  {p:>5} -> {v.astype(int)}")


The ordinal version reads as icmp 0, tcp 1, udp 2. A linear model or a
distance-based model now believes udp is twice tcp and that icmp is at one end
of a scale, because that is what those numbers mean. The scikit-learn guide puts
it plainly: integer codes "would interpret the categories as being ordered,
which is often not desired".

One-hot gives each category its own column and no ordering at all. It is the
default choice for categories with no natural sequence.

Ordinal is the right answer when an order genuinely exists and you choose it
deliberately: low, medium, high, or severity 1 to 5. The failure is not using
ordinal encoding. It is using it by accident and inventing a ranking the data
never had.


## What to take from this

| Finding | Measured |
|---|---|
| The column you pick decides the outcome | Iris, one column at a time: 0.553 against 0.960 |
| Every summary you compute costs something | Digits, 64 features to 1: 0.969 down to 0.125 |
| Units are part of the feature | Wine, k-nearest neighbours: 0.663 raw against 0.961 scaled |
| Whether scale matters depends on the model | Wine, decision tree: unchanged by scaling |
| Encoding a category invents structure if you are careless | Ordinal makes udp twice tcp |

One habit follows from all five. Before reaching for a better model, write down
what each column actually is, what you computed it from, and what you discarded
to get it. That document is the difference between a model you can explain and a
model you can only report.

And if a feature scores suspiciously well, do not celebrate. Lesson 2.3 is about
exactly that.
